# Zynq PS–PL INT16 GEMM Integration

이 notebook은 8×8 INT16 GEMM accelerator를 Zynq PS에서 구동하고 검증한 작업을 요약한 것입니다.

다루는 범위는 **PS–PL 역할 분리, 검증 방법, 확인된 정합성 결과**입니다. Bitstream, model weight, DMA register sequence, weight layout 변환, buffer-pool 구현과 전체 inference 코드는 공개하지 않습니다.

## System boundary

```text
Zynq PS
  ├─ input quantization
  ├─ accelerator configuration
  ├─ DMA buffer management
  ├─ PL execution request / completion wait
  └─ CPU quantized-reference comparison
          │
          ▼
Zynq PL
  └─ 8×8 INT16 GEMM accelerator
```

PL RTL은 외부 공개 프로젝트를 기반으로 하며 이 저장소에 포함하지 않습니다. 이 notebook은 PS-side integration과 verification을 다룹니다.

## PS-side work

- INT16 accelerator용 입력/출력 buffer lifecycle 관리
- Matrix dimension과 accelerator tile constraint 검사
- Feature, weight, result DMA의 실행 순서 관리
- Cache flush/invalidate가 필요한 PS–PL shared-memory boundary 처리
- CPU quantized reference와 PL output의 numerical equivalence 검증
- Prefill과 single-token decode에서 KV-cache shape transition 확인
- End-to-end latency와 GEMM-level latency를 분리하는 profiling 설계


In [ ]:
from dataclasses import dataclass
from typing import Protocol
import numpy as np


@dataclass(frozen=True)
class MatchMetrics:
    max_abs_diff: float
    mean_abs_diff: float
    rmse: float
    cosine_similarity: float
    same_argmax: bool
    topk_overlap: int


def compare_logits(reference, candidate, k=10):
    reference = np.asarray(reference, dtype=np.float32).reshape(-1)
    candidate = np.asarray(candidate, dtype=np.float32).reshape(-1)
    if reference.shape != candidate.shape:
        raise ValueError(f'shape mismatch: {reference.shape} != {candidate.shape}')

    diff = reference - candidate
    denom = np.linalg.norm(reference) * np.linalg.norm(candidate) + 1e-12
    ref_topk = np.argsort(reference)[-k:]
    cand_topk = np.argsort(candidate)[-k:]

    return MatchMetrics(
        max_abs_diff=float(np.max(np.abs(diff))),
        mean_abs_diff=float(np.mean(np.abs(diff))),
        rmse=float(np.sqrt(np.mean(diff ** 2))),
        cosine_similarity=float(np.dot(reference, candidate) / denom),
        same_argmax=bool(np.argmax(reference) == np.argmax(candidate)),
        topk_overlap=len(set(ref_topk.tolist()) & set(cand_topk.tolist())),
    )

In [ ]:
class Int16GemmBackend(Protocol):
    """Public interface boundary; the DMA/MMIO implementation is withheld."""

    def run(self, activation_q: np.ndarray, weight_q: np.ndarray, *, shift: int) -> np.ndarray:
        ...


def validate_projection(backend, activation_q, weight_q, shift, reference_logits):
    """Compare PL output against an independently computed quantized reference."""
    pl_logits = backend.run(activation_q, weight_q, shift=shift)
    return compare_logits(reference_logits, pl_logits, k=10)

## Recorded verification evidence

아래 값은 원본 PYNQ 실행 notebook에 저장된 결과에서 옮긴 것입니다. Hardware correctness는 float model이 아니라 **CPU INT16 quantized reference**를 기준으로 판단했습니다.

| Test | Max abs diff | Mean abs diff | RMSE | Same argmax | Top-10 overlap |
|---|---:|---:|---:|:---:|:---:|
| Prefill: CPU quant vs PL | 6.4373e-6 | 9.2476e-7 | 1.2232e-6 | Yes | 10/10 |
| Decode 1-step: CPU quant vs PL | 1.0490e-5 | 1.7173e-6 | 2.1043e-6 | Yes | 10/10 |

Top-10 overlap은 상위 10개 후보 집합의 겹침이며 순위 일치를 뜻하지 않습니다.

## Interpretation

- CPU quantized reference와 PL 결과가 근접하므로 PS–PL data layout과 accelerator invocation의 정합성을 확인했습니다.
- Float reference와 INT16 quantized reference 사이에는 model-level accuracy drift가 존재했습니다. 이는 PL 계산 오류와 분리해서 해석해야 합니다.
- 원본 notebook의 상세 benchmark cell은 실행되지 않았으므로 여기서는 speedup이나 throughput 수치를 주장하지 않습니다.
- 저장된 단일 실행 시간은 정식 benchmark가 아니므로 성능 지표에서 제외했습니다.

## Not included

- DMA register offsets and transaction sequence
- Exact tensor packing and weight-transpose implementation
- Quantization-scale and shift-selection implementation
- Persistent DDR weight-buffer and shared buffer-pool implementation
- Full transformer layer, prefill, decode and KV-cache source
- Per-stage profiling implementation and projected-speedup calculation
- Bitstream, model weights, tokenizer and vocabulary
- Raw generated text and token IDs


## Reproduction boundary

이 notebook은 architecture와 verification evidence를 정리한 문서이며 standalone executable package가 아닙니다. 재실행에는 PYNQ board, 호환 bitstream, PS driver, quantized model assets가 필요합니다.

No license is granted. All rights reserved.